# T2-11 Colab GPU indexing (KURE-v1)

Embeds `data/tables.jsonl` (table metadata only, no values) on a Colab GPU and writes one `.npz` build.
Load it into your local Chroma with `python -m kosis_rag.index --import <file>.npz` (no embedding on your machine).

1. Runtime -> Change runtime type -> **T4 GPU** (or better).
2. Run the cells in order; upload `data/tables.jsonl` from your machine when asked.
3. Delete the uploaded file from the runtime afterwards (last cell).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 --recurse-submodules --shallow-submodules https://github.com/addiescode-sj/KOSIS-ai-RAG.git repo
!pip -q install sentence-transformers python-dotenv

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('repo/data', exist_ok=True)
uploaded = files.upload()  # pick data/tables.jsonl
shutil.move(next(iter(uploaded)), 'repo/data/tables.jsonl')

In [ ]:
%cd repo
!PYTHONPATH=.:src:vendor/phase1:vendor/phase1/src python -m kosis_rag.index --device cuda --export kure_build.npz

In [ ]:
files.download('kure_build.npz')

In [ ]:
# The provided data must not stay on the runtime
os.remove('data/tables.jsonl'); os.remove('kure_build.npz')
print('removed')

Then on your machine (repo root, Chroma running):

```bash
docker compose up -d chroma
PYTHONPATH=.:src:vendor/phase1:vendor/phase1/src uv run python -m kosis_rag.index --import ~/Downloads/kure_build.npz
```

Put the printed `CHROMA_COLLECTION=...` in `.env`.